# 12주차 Type III 다중회귀분석 Colab 실습노트

이 실습노트는 Type III에서 자주 등장하는 **다중회귀분석**을 Google Colab에서 직접 실행하고 해석할 수 있도록 구성한 자료입니다.

이번 주차의 핵심은 다음 2가지입니다.

1. **회귀계수·유의성·R² 해석**
   - 파일: `retail_sales_reg1.csv`
   - 주제: 월매출(`sales`)에 영향을 미치는 광고비(`ad_cost`), 직원 수(`staff`), 주말행사 횟수(`event_cnt`) 분석

2. **예측값·신뢰구간·오차지표 해석**
   - 파일: `weather_reg2.csv`
   - 주제: 기온(`temperature`)을 설명하는 일사량(`solar`), 바람(`wind`), 오존(`o3`) 분석

---

## 학습 목표
- 회귀식을 구성할 수 있다.
- 회귀계수의 의미를 설명할 수 있다.
- p-value를 보고 유의한 변수를 판별할 수 있다.
- 결정계수(R²)의 의미를 설명할 수 있다.
- 예측값, 신뢰구간, MSE/RMSE를 코드와 연결해서 해석할 수 있다.

---

## 핵심 Python method
- `pd.read_csv()`
- `statsmodels.formula.api.ols()`
- `.fit()`
- `.summary()`
- `.params`
- `.pvalues`
- `.rsquared`
- `.predict()`
- `.get_prediction().summary_frame()`
- `mean_squared_error()`

## 0. Colab 시작 설정

아래 셀을 먼저 실행하세요.

- Google Drive 마운트
- CSV 경로 설정
- 파일 존재 여부 확인

In [7]:
# 0-1. Google Drive 마운트 및 라이브러리 로드
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error

# 드라이브 연결
drive.mount('/content/drive')

# 데이터 폴더 경로 설정
# 예시: MyDrive/type3_week12
DATA_DIR = Path('/content/drive/MyDrive/type3_week1')
print('설정된 데이터 폴더:', DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
설정된 데이터 폴더: /content/drive/MyDrive/type3_week1


In [8]:
# 0-2. 파일 존재 여부 확인
required_files = ['retail_sales_reg1.csv', 'weather_reg2.csv']

print('[파일 존재 여부 확인]')
for fname in required_files:
    fpath = DATA_DIR / fname
    print(f'{fname}:', 'OK' if fpath.exists() else '없음')

[파일 존재 여부 확인]
retail_sales_reg1.csv: OK
weather_reg2.csv: OK


---
# 실습 1. `retail_sales_reg1.csv`
## 다중회귀 01: 회귀계수·유의성·R² 해석

## 1-1. 도입 설명

한 소매업체의 월매출(`sales`)에 영향을 미치는 요인을 분석하려고 합니다.
설명변수로는 다음 3개를 사용합니다.

- 광고비: `ad_cost`
- 직원 수: `staff`
- 주말행사 횟수: `event_cnt`

이 실습에서 학생들은 다음을 확인해야 합니다.

1. 회귀식은 어떻게 만들어지는가?
2. 각 회귀계수는 무엇을 의미하는가?
3. p-value를 보고 어떤 변수가 유의한지 어떻게 판단하는가?
4. R²는 무엇을 의미하는가?

---

## 1-2. 분석 전에 기억할 것

- 회귀계수는 **다른 변수가 고정되었을 때**, 해당 변수가 1 증가하면 종속변수가 평균적으로 얼마나 변하는지를 의미합니다.
- p-value < 0.05 이면 보통 **통계적으로 유의한 변수**라고 판단합니다.
- R²는 모델이 종속변수 변동을 얼마나 설명하는지 보여줍니다.

In [9]:
# 1-3. 데이터 불러오기
file_path = DATA_DIR / 'retail_sales_reg1.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())
print('[기초통계]')
print(df.describe())

[데이터 상위 5행]
    sales  ad_cost  staff  event_cnt
0  344.42    38.61     12          4
1  316.86    49.40     14          3
2  290.08    49.31      5          4
3  356.27    46.20     15          4
4  273.46    50.36     13          2
[기본 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sales      80 non-null     float64
 1   ad_cost    80 non-null     float64
 2   staff      80 non-null     int64  
 3   event_cnt  80 non-null     int64  
dtypes: float64(2), int64(2)
memory usage: 2.6 KB
None
[기초통계]
            sales    ad_cost      staff  event_cnt
count   80.000000  80.000000  80.000000  80.000000
mean   326.516250  63.250000  12.337500   1.912500
std     76.493853  22.107343   5.279393   1.370519
min    155.950000  26.760000   5.000000   0.000000
25%    274.217500  45.940000   7.000000   1.000000
50%    329.670000  65.495000  13.000000   2.0000

In [10]:
# 1-4. 다중회귀모형 적합
# sales ~ ad_cost + staff + event_cnt
model = smf.ols('sales ~ ad_cost + staff + event_cnt', data=df).fit()

print('[회귀 요약 결과]')
print(model.summary())

[회귀 요약 결과]
                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.886
Model:                            OLS   Adj. R-squared:                  0.881
Method:                 Least Squares   F-statistic:                     196.5
Date:                Mon, 25 May 2026   Prob (F-statistic):           1.03e-35
Time:                        16:55:55   Log-Likelihood:                -373.20
No. Observations:                  80   AIC:                             754.4
Df Residuals:                      76   BIC:                             763.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     57.8033     11.716      4.9

In [12]:
# 1-5. 핵심 결과만 따로 보기
print('[회귀계수]')
print(model.params)

print('[p-value]')
print(model.pvalues)

print('[R-squared]')
print(model.rsquared)

print('[95% 신뢰구간]')
print(model.conf_int())

[회귀계수]
Intercept    57.803344
ad_cost       2.797464
staff         4.263380
event_cnt    20.483058
dtype: float64
[p-value]
Intercept    4.637357e-06
ad_cost      5.448824e-33
staff        1.043095e-10
event_cnt    3.196052e-14
dtype: float64
[R-squared]
0.8858042080792261
[95% 신뢰구간]
                   0          1
Intercept  34.469209  81.137478
ad_cost     2.528502   3.066426
staff       3.129470   5.397290
event_cnt  16.108057  24.858058


In [13]:
# 1-6. 광고비 10단위 증가 효과 해석 예시
beta_ad = model.params['ad_cost']
change_ad_10 = beta_ad * 10

print('광고비 계수:', beta_ad)
print('광고비 10단위 증가 시 예상 매출 변화:', change_ad_10)

# 유의한 변수 목록
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print('유의한 변수:', sig_vars)

광고비 계수: 2.7974641115899317
광고비 10단위 증가 시 예상 매출 변화: 27.974641115899317
유의한 변수: ['Intercept', 'ad_cost', 'staff', 'event_cnt']


## 1-7. 출력 결과 해석 가이드

아래 순서로 읽게 지도하면 좋습니다.

1. **회귀식 확인**
   - 절편과 각 변수 계수를 읽습니다.
2. **p-value 확인**
   - 0.05보다 작은 변수는 유의하다고 판단합니다.
3. **R² 확인**
   - 모델 설명력을 해석합니다.
4. **효과 크기 해석**
   - 예: 광고비가 10 증가하면 매출은 얼마나 증가하는가?

### 해석 문장 예시
- `ad_cost`의 계수가 양수이면 광고비가 증가할수록 매출이 증가하는 경향이 있다고 해석합니다.
- `event_cnt`의 p-value가 0.05보다 작으면 주말행사 횟수는 매출에 유의한 영향을 준다고 해석합니다.
- R²가 높을수록 모델이 매출 변동을 잘 설명한다고 볼 수 있습니다.

## 1-8. 학생 확인 질문

1. 회귀식에서 종속변수는 무엇인가?
2. `ad_cost`의 계수 부호는 무엇이며, 그 의미는 무엇인가?
3. `staff`의 p-value가 0.05보다 작다면 어떤 의미인가?
4. R² 값이 0.88이라면, 이 모델은 무엇을 88% 정도 설명한다고 볼 수 있는가?
5. 광고비가 10단위 증가할 때 매출 변화량을 어떻게 계산하는가?

---
# 실습 2. `weather_reg2.csv`
## 다중회귀 02: 예측값·신뢰구간·오차지표 해석

## 2-1. 도입 설명

환경 데이터 분석팀은 기온(`temperature`)을 설명하기 위해 아래 설명변수를 사용합니다.

- 일사량: `solar`
- 바람: `wind`
- 오존: `o3`

이 실습에서는 단순히 회귀계수만 보는 것이 아니라,

1. 특정 조건에서의 **예측값**
2. **신뢰구간**
3. **MSE/RMSE**
4. 모델의 설명력(R²)

까지 함께 해석하는 연습을 합니다.

---

## 2-2. 분석 전에 기억할 것

- `predict()`는 예측값을 계산합니다.
- `get_prediction().summary_frame()`은 예측값과 함께 신뢰구간 관련 정보를 제공합니다.
- MSE는 오차의 제곱 평균, RMSE는 그 제곱근입니다.
- RMSE는 원래 종속변수와 같은 단위이므로 해석이 더 직관적입니다.

In [14]:
# 2-3. 데이터 불러오기
file_path = DATA_DIR / 'weather_reg2.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())
print('[기초통계]')
print(df.describe())

[데이터 상위 5행]
   temperature   solar  wind     o3
0        14.45   61.16  4.12  16.67
1        17.75  108.21  7.18  30.84
2        20.31  123.01  9.71  45.34
3        20.33  172.55  9.62  53.31
4        18.69  196.61  5.82  25.85
[기본 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   temperature  150 non-null    float64
 1   solar        150 non-null    float64
 2   wind         150 non-null    float64
 3   o3           150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB
None
[기초통계]
       temperature       solar        wind          o3
count   150.000000  150.000000  150.000000  150.000000
mean     21.966333  180.878133    6.131733   32.380133
std       6.368432   72.173137    3.335212   16.213730
min       2.590000   51.130000    1.010000    5.130000
25%      17.370000  124.252500    3.100000   18.415000
50%      22.130000  183.86

In [15]:
# 2-4. 다중회귀모형 적합
model = smf.ols('temperature ~ solar + wind + o3', data=df).fit()

print('[회귀 요약 결과]')
print(model.summary())

[회귀 요약 결과]
                            OLS Regression Results                            
Dep. Variable:            temperature   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.879
Method:                 Least Squares   F-statistic:                     362.3
Date:                Mon, 25 May 2026   Prob (F-statistic):           2.10e-67
Time:                        16:56:32   Log-Likelihood:                -330.04
No. Observations:                 150   AIC:                             668.1
Df Residuals:                     146   BIC:                             680.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     11.7807      0.729     16.1

In [16]:
# 2-5. 핵심 결과만 따로 보기
print('[회귀계수]')
print(model.params)

print('[p-value]')
print(model.pvalues)

print('[R-squared]')
print(model.rsquared)

print('[95% 신뢰구간]')
print(model.conf_int())

[회귀계수]
Intercept    11.780716
solar         0.043943
wind         -0.921975
o3            0.243684
dtype: float64
[p-value]
Intercept    2.362300e-34
solar        1.582367e-37
wind         3.260190e-36
o3           1.494907e-47
dtype: float64
[R-squared]
0.8815680127071354
[95% 신뢰구간]
                   0          1
Intercept  10.340886  13.220546
solar       0.038964   0.048923
wind       -1.029717  -0.814232
o3          0.221497   0.265870


In [18]:
# 2-6. 특정 조건에서의 예측값 계산
new_df = pd.DataFrame({
    'solar': [100],
    'wind': [5],
    'o3': [30]
})

pred = model.predict(new_df)
print('[예측값]')
print(pred)

[예측값]
0    18.875697
dtype: float64


In [19]:
# 2-7. 예측 요약표(평균 예측과 구간 정보)
pred_frame = model.get_prediction(new_df).summary_frame(alpha=0.05)

print('[예측 요약표]')
print(pred_frame)

[예측 요약표]
        mean   mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  \
0  18.875697  0.283824      18.314764      19.436631     14.464199   

   obs_ci_upper  
0     23.287195  


In [20]:
# 2-8. 오차지표 계산
# 실제값과 모델 예측값 비교

y_true = df['temperature']
y_pred = model.predict(df)

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

print('[오차지표]')
print('MSE :', mse)
print('RMSE:', rmse)

[오차지표]
MSE : 4.771215059149032
RMSE: 2.184311117755214


## 2-9. 출력 결과 해석 가이드

아래 순서로 읽게 지도하면 좋습니다.

1. **계수와 부호 해석**
   - `solar`가 양수이면 일사량이 증가할수록 기온이 상승하는 방향
   - `wind`가 음수이면 바람이 강할수록 기온이 하락하는 방향
2. **p-value 확인**
   - 0.05보다 작으면 유의한 설명변수
3. **예측값 해석**
   - 예: `solar=100, wind=5, o3=30`에서 예상 기온은 얼마인가?
4. **신뢰구간 확인**
   - 평균 예측에 대한 불확실성 범위를 확인
5. **MSE / RMSE 해석**
   - RMSE는 평균 예측 오차 크기를 원래 단위로 보여줌

### 해석 문장 예시
- `wind`의 회귀계수가 음수이므로, 다른 조건이 같을 때 바람이 강할수록 기온은 감소하는 경향이 있다.
- `o3`의 p-value가 매우 작다면, 오존은 기온 예측에 유의한 변수라고 해석할 수 있다.
- RMSE가 작을수록 모델 예측은 실제값에 더 가깝다.

## 2-10. 학생 확인 질문

1. `temperature`를 설명하는 변수는 무엇인가?
2. `wind`의 계수가 음수라면 어떤 의미인가?
3. 특정 조건(`solar=100, wind=5, o3=30`)에서 예측값은 무엇을 의미하는가?
4. `get_prediction().summary_frame()`은 단순 `predict()`와 무엇이 다른가?
5. RMSE는 왜 MSE보다 해석이 쉬운가?

---
# 마무리 정리

## 오늘 실습에서 기억해야 할 것

### 1. 다중회귀는 언제 쓰는가?
- **여러 설명변수**가 **연속형 종속변수**에 미치는 영향을 동시에 보고 싶을 때 사용합니다.

### 2. 가장 중요한 출력값
- 회귀계수(`params`)
- p-value(`pvalues`)
- 결정계수(`rsquared`)
- 예측값(`predict`)
- 신뢰구간(`get_prediction().summary_frame()`)
- 오차지표(MSE, RMSE)

### 3. 보고서에 꼭 들어가야 할 것
- 회귀식
- 유의한 변수 여부
- R² 해석
- 예측값 설명
- RMSE 해석

---

## 과제 제안
1. 두 데이터셋에 대해 회귀식과 유의한 변수를 직접 정리해 보기
2. `retail_sales_reg1.csv`에서 가장 영향력이 큰 변수를 설명해 보기
3. `weather_reg2.csv`에서 새로운 조건을 하나 더 만들어 예측값을 구해 보기
4. RMSE가 무엇을 의미하는지 한 문장으로 정리해 보기